# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

data_files = sorted(PROCESSED_DIR.glob("*_data.csv"))
target_files = {path.name.replace("_targets.csv", ""): path for path in PROCESSED_DIR.glob("*_targets.csv")}

print(f"Found {len(data_files)} processed data files.")
print(f"Saving figures to: {IMAGES_DIR}")

In [ ]:
MAX_LAG = 150  # seconds
DELTA = 20    # PRBS interval duration
Z = 15        # number of PRBS intervals
C_CONST = 1 + 1 / Z
PRBS_START_SEC = 300
PRBS_END_SEC = 1000

def find_column(df: pd.DataFrame, target: str) -> str:
    lowered = target.lower()
    for column in df.columns:
        if column.lower() == lowered:
            return column
    for column in df.columns:
        if column.lower().startswith(lowered):
            return column
    for column in df.columns:
        if lowered in column.lower():
            return column
    raise KeyError(f"Column '{target}' not found in dataframe.")

def ensure_one_hz(time: np.ndarray, signals: list[np.ndarray]) -> tuple[np.ndarray, list[np.ndarray]]:
    time = np.asarray(time, dtype=float)
    signals = [np.asarray(sig, dtype=float) for sig in signals]

    finite_mask = np.isfinite(time)
    for sig in signals:
        finite_mask &= np.isfinite(sig)
    time = time[finite_mask]
    signals = [sig[finite_mask] for sig in signals]
    if time.size < 2:
        raise ValueError("Insufficient samples to enforce 1 Hz grid.")

    order = np.argsort(time)
    time = time[order]
    signals = [sig[order] for sig in signals]

    time = time - np.nanmin(time)
    diffs = np.diff(time)
    valid_diffs = diffs[(diffs > 0) & np.isfinite(diffs)]
    dt = np.median(valid_diffs) if valid_diffs.size else 1.0
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Unable to determine sampling interval for 1 Hz resampling.")

    if abs(dt - 1.0) > 1e-2:
        end_time = time[-1]
        resample_times = np.arange(0.0, end_time + 1.0, 1.0)
        resampled = [np.interp(resample_times, time, sig) for sig in signals]
        return resample_times, resampled
    return time, signals

def compute_cross_correlation(x: np.ndarray, y: np.ndarray, max_lag: int) -> dict[int, float]:
    x_centered = np.asarray(x, dtype=float) - np.nanmean(x)
    y_centered = np.asarray(y, dtype=float) - np.nanmean(y)
    corr = signal.correlate(y_centered, x_centered, mode="full")
    lags = signal.correlation_lags(len(y_centered), len(x_centered), mode="full")
    mask = (lags >= -max_lag) & (lags <= max_lag)
    corr = corr[mask]
    lags = lags[mask]
    return {int(lag): float(value) for lag, value in zip(lags, corr)}

def phi_at(phi: dict[int, float], lag: int) -> float:
    return phi.get(int(lag), 0.0)

def compute_impulse_response_from_ccf(phi: dict[int, float], max_lag: int, delta: int, C: float) -> np.ndarray:
    h_tilde = np.zeros(max_lag, dtype=float)
    for t in range(max_lag):
        phi_pos = phi_at(phi, t)
        phi_neg = phi_at(phi, -t)
        if t == 0:
            phi_neg_delta = phi_at(phi, -delta)
            h_tilde[t] = 2.0 * (phi_pos - phi_neg_delta) / C
        elif t < delta:
            denom = (0.5 + (t / delta) * (1 - t / (2 * delta))) * C
            if abs(denom) < 1e-12:
                h_tilde[t] = 0.0
            else:
                h_tilde[t] = (phi_pos - phi_neg) / denom
        else:
            h_tilde[t] = (phi_pos - phi_neg) / C
    total = np.sum(h_tilde)
    if abs(total) < 1e-12:
        raise ValueError("Impulse response sum is zero; cannot normalise.")
    return h_tilde / total

def fit_gain_offset(prbs_prediction: np.ndarray, vo2: np.ndarray) -> tuple[float, float]:
    A = np.vstack([prbs_prediction, np.ones_like(prbs_prediction)]).T
    coeffs, *_ = np.linalg.lstsq(A, vo2, rcond=None)
    return float(coeffs[0]), float(coeffs[1])

def load_subject_signals(data_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_csv(data_path)
    time_col = find_column(df, "Time")
    wr_col = find_column(df, "WorkRate")
    vo2_col = find_column(df, "VO2pulm")
    subset = df[[time_col, wr_col, vo2_col]].dropna()
    if subset.empty:
        raise ValueError("No valid samples after dropping NaNs.")
    time = subset[time_col].to_numpy(dtype=float)
    wr = subset[wr_col].to_numpy(dtype=float)
    vo2 = subset[vo2_col].to_numpy(dtype=float)
    time_aligned, [wr_aligned, vo2_aligned] = ensure_one_hz(time, [wr, vo2])
    return time_aligned, wr_aligned, vo2_aligned

def load_actual_vo2peak(target_path: Path) -> float:
    targets_df = pd.read_csv(target_path)
    column = find_column(targets_df, "VO2peak_abs")
    return float(targets_df[column].iloc[0])

In [ ]:
results = []
residuals = []

for data_path in data_files:
    subject_id = data_path.name.replace("_data.csv", "")
    target_path = target_files.get(subject_id)
    if target_path is None:
        print(f"{subject_id}: missing targets file; skipping.")
        continue
    try:
        time_series, work_rate, vo2 = load_subject_signals(data_path)
    except Exception as exc:
        print(f"{subject_id}: failed to load signals ({exc}); skipping.")
        continue

    prbs_mask = (time_series >= PRBS_START_SEC) & (time_series <= PRBS_END_SEC)
    if prbs_mask.sum() <= MAX_LAG:
        print(f"{subject_id}: insufficient PRBS samples in 300-1000 s window; skipping.")
        continue
    time_prbs = time_series[prbs_mask]
    work_rate_prbs = work_rate[prbs_mask]
    vo2_prbs = vo2[prbs_mask]
    time_prbs = time_prbs - time_prbs[0]

    try:
        phi = compute_cross_correlation(work_rate_prbs, vo2_prbs, MAX_LAG)
        impulse_response = compute_impulse_response_from_ccf(phi, MAX_LAG, DELTA, C_CONST)
    except Exception as exc:
        print(f"{subject_id}: impulse response failure ({exc}); skipping.")
        continue

    prbs_conv = signal.convolve(work_rate_prbs, impulse_response, mode="full")[: len(work_rate_prbs)]
    try:
        gain, offset = fit_gain_offset(prbs_conv, vo2_prbs)
    except np.linalg.LinAlgError as exc:
        print(f"{subject_id}: gain/offset fit failed ({exc}); skipping.")
        continue

    ramp_input = np.linspace(80.0, 300.0, 301)
    ramp_conv = signal.convolve(ramp_input, impulse_response, mode="full")[: len(ramp_input)]
    ramp_vo2_pred = gain * ramp_conv + offset
    vo2peak_pred = float(np.nanmax(ramp_vo2_pred))
    try:
        vo2peak_actual = load_actual_vo2peak(target_path)
    except Exception as exc:
        print(f"{subject_id}: target load failed ({exc}); skipping.")
        continue

    residual_series = vo2_prbs - (gain * prbs_conv + offset)
    residuals.extend(residual_series.tolist())

    results.append({
        "subject": subject_id,
        "predicted_vo2peak": vo2peak_pred,
        "actual_vo2peak": vo2peak_actual,
        "gain": gain,
        "offset": offset
    })
    print(f"{subject_id}: predicted {vo2peak_pred:.3f} L/min, actual {vo2peak_actual:.3f} L/min")

In [ ]:
results_df = pd.DataFrame(results)
if results_df.empty:
    raise RuntimeError("No subjects processed successfully. Confirm data preprocessing steps.")
results_df

In [ ]:
if "actual_list" not in globals():
    actual_list = results_df["actual_vo2peak"].tolist()
if "predicted_list" not in globals():
    predicted_list = results_df["predicted_vo2peak"].tolist()

actual_arr = np.asarray(actual_list, dtype=float)
predicted_arr = np.asarray(predicted_list, dtype=float)

mae = mean_absolute_error(actual_arr, predicted_arr)
rmse = np.sqrt(mean_squared_error(actual_arr, predicted_arr))
r2 = r2_score(actual_arr, predicted_arr)

B = 1000
n_samples = len(actual_arr)
bootstrap_mae = np.empty(B)
bootstrap_rmse = np.empty(B)
bootstrap_r2 = np.empty(B)

pairs = np.column_stack([actual_arr, predicted_arr])
for i in range(B):
    sample = resample(pairs, replace=True, n_samples=n_samples)
    sample_actual = sample[:, 0]
    sample_predicted = sample[:, 1]
    bootstrap_mae[i] = mean_absolute_error(sample_actual, sample_predicted)
    bootstrap_rmse[i] = np.sqrt(mean_squared_error(sample_actual, sample_predicted))
    bootstrap_r2[i] = r2_score(sample_actual, sample_predicted)

mae_ci = np.percentile(bootstrap_mae, [2.5, 97.5])
rmse_ci = np.percentile(bootstrap_rmse, [2.5, 97.5])
r2_ci = np.percentile(bootstrap_r2, [2.5, 97.5])

print(f"MAE: {mae:.3f} L/min [{mae_ci[0]:.3f}, {mae_ci[1]:.3f}]")
print(f"RMSE: {rmse:.3f} L/min [{rmse_ci[0]:.3f}, {rmse_ci[1]:.3f}]")
print(f"R^2: {r2:.3f} [{r2_ci[0]:.3f}, {r2_ci[1]:.3f}]")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(results_df['actual_vo2peak'], results_df['predicted_vo2peak'], color='royalblue', alpha=0.75)
lims = [
    min(results_df['actual_vo2peak'].min(), results_df['predicted_vo2peak'].min()),
    max(results_df['actual_vo2peak'].max(), results_df['predicted_vo2peak'].max())
 ]
plt.plot(lims, lims, '--', color='gray', label='Ideal 1:1')
plt.xlabel('Actual VO2peak_abs (L/min)')
plt.ylabel('Predicted VO2peak_abs (L/min)')
plt.title('Impulse Response VO2peak Predictions')
plt.xlim(lims)
plt.ylim(lims)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'impulse_response_pred_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(residuals, bins=30, color='teal', alpha=0.7)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Distribution (PRBS fit)')
plt.xlabel('VO2 residual (L/min)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'impulse_response_residual_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# add Bland–Altman plot

mean_vo2 = (results_df['actual_vo2peak'] + results_df['predicted_vo2peak']) / 2
diff_vo2 = results_df['predicted_vo2peak'] - results_df['actual_vo2peak']
mean_diff = np.mean(diff_vo2)
std_diff = np.std(diff_vo2)

plt.figure(figsize=(6, 4))
plt.scatter(mean_vo2, diff_vo2, color='purple', alpha=0.7)
plt.axhline(mean_diff, color='red', linestyle='--', label='Mean Difference')
plt.axhline(mean_diff + 1.96 * std_diff, color='gray', linestyle='--', label='+1.96 SD')
plt.axhline(mean_diff - 1.96 * std_diff, color='gray', linestyle='--', label='-1.96 SD')
plt.xlabel('Mean VO2peak_abs (L/min)')
plt.ylabel('Difference in VO2peak_abs (Predicted - Actual) (L/min)')
plt.title('Bland–Altman Plot for VO2peak Predictions')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'impulse_response_bland_altman.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
subject_plot_data = None
for data_path in data_files:
    subject_id = data_path.name.replace("_data.csv", "")
    target_path = target_files.get(subject_id)
    if target_path is None:
        continue
    try:
        time_series, work_rate, vo2 = load_subject_signals(data_path)
        prbs_mask = (time_series >= PRBS_START_SEC) & (time_series <= PRBS_END_SEC)
        if prbs_mask.sum() <= MAX_LAG:
            continue
        time_prbs = time_series[prbs_mask]
        work_rate_prbs = work_rate[prbs_mask]
        vo2_prbs = vo2[prbs_mask]
        time_prbs = time_prbs - time_prbs[0]
        phi = compute_cross_correlation(work_rate_prbs, vo2_prbs, MAX_LAG)
        impulse_response = compute_impulse_response_from_ccf(phi, MAX_LAG, DELTA, C_CONST)
        prbs_conv = signal.convolve(work_rate_prbs, impulse_response, mode="full")[: len(work_rate_prbs)]
        gain, offset = fit_gain_offset(prbs_conv, vo2_prbs)
        ramp_input = np.linspace(80.0, 300.0, 301)
        ramp_conv = signal.convolve(ramp_input, impulse_response, mode="full")[: len(ramp_input)]
        ramp_vo2_pred = gain * ramp_conv + offset
        subject_plot_data = {
            "subject": subject_id,
            "time": time_prbs,
            "work_rate": work_rate_prbs,
            "vo2": vo2_prbs,
            "impulse": impulse_response,
            "vo2_prbs_pred": gain * prbs_conv + offset,
            "ramp_time": np.arange(len(ramp_vo2_pred), dtype=float),
            "vo2_ramp_pred": ramp_vo2_pred
        }
        break
    except Exception as exc:
        print(f"{subject_id}: skipping plot data due to {exc}")
        continue
if subject_plot_data is None:
    raise RuntimeError("Failed to build plot inputs for any subject.")

In [ ]:
plot_time = subject_plot_data['time']
plot_work_rate = subject_plot_data['work_rate']
plot_vo2 = subject_plot_data['vo2']
plot_impulse = subject_plot_data['impulse']
plot_vo2_prbs = subject_plot_data['vo2_prbs_pred']
plot_ramp_time = subject_plot_data['ramp_time']
plot_vo2_ramp = subject_plot_data['vo2_ramp_pred']
plot_subject = subject_plot_data['subject']

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.ravel()

axes[0].plot(plot_time, plot_work_rate, color='tab:orange')
axes[0].set_title(f"{plot_subject} Work Rate PRBS")
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('WR (W)')
axes[0].grid(True, linestyle='--', alpha=0.4)

axes[1].plot(np.arange(len(plot_impulse)), plot_impulse, color='tab:green')
axes[1].set_title('Impulse Response h(t)')
axes[1].set_xlabel('Lag (s)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, linestyle='--', alpha=0.4)

axes[2].plot(plot_time, plot_vo2, label='Actual VO2', color='tab:blue')
axes[2].plot(plot_time, plot_vo2_prbs, label='Predicted VO2 (PRBS)', color='tab:red', alpha=0.8)
axes[2].set_title('PRBS VO2 Fit')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('VO2 (L/min)')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.4)

axes[3].axis('off')

axes[4].plot(plot_ramp_time, np.linspace(80.0, 300.0, len(plot_ramp_time)), color='tab:orange')
axes[4].set_title('Ramp Input')
axes[4].set_xlabel('Time (s)')
axes[4].set_ylabel('WR (W)')
axes[4].grid(True, linestyle='--', alpha=0.4)

axes[5].plot(plot_ramp_time, plot_vo2_ramp, color='tab:red')
axes[5].set_title('Ramp VO2 Prediction')
axes[5].set_xlabel('Time (s)')
axes[5].set_ylabel('VO2 (L/min)')
axes[5].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
figure_name = f"impulse_response_subject_overview_{plot_subject}.png"
plt.savefig(IMAGES_DIR / figure_name, dpi=300, bbox_inches='tight')
plt.show()